# Submission - Predicción de Exam Scores

## Modelo
- CatBoost optimizado con Optuna
- Variables: study_hours, sleep_quality, facility_rating, class_attendance, study_method

## Archivos
- Input: data/test.csv
- Output: submission_v2.csv

In [1]:
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor

print("✓ Librerías cargadas")

✓ Librerías cargadas


In [2]:
# Rutas de archivos
BASE_PATH = r"C:\Users\HP\OneDrive\Escritorio\David Guzzi\Github\DGKaggle\Playground Series\Season 6\Episode 1 - Predicting Student Test Scores"
MODEL_PATH = f"{BASE_PATH}/notebooks/best_catboost_model_v2.cbm"
TEST_PATH = f"{BASE_PATH}/data/test.csv"
SAMPLE_PATH = f"{BASE_PATH}/data/sample_submission.csv"
OUTPUT_PATH = f"{BASE_PATH}/notebooks/submission_v2.csv"

print("Rutas configuradas:")
print(f"  Modelo: {MODEL_PATH}")
print(f"  Test:   {TEST_PATH}")
print(f"  Output: {OUTPUT_PATH}")

Rutas configuradas:
  Modelo: C:\Users\HP\OneDrive\Escritorio\David Guzzi\Github\DGKaggle\Playground Series\Season 6\Episode 1 - Predicting Student Test Scores/notebooks/best_catboost_model_v2.cbm
  Test:   C:\Users\HP\OneDrive\Escritorio\David Guzzi\Github\DGKaggle\Playground Series\Season 6\Episode 1 - Predicting Student Test Scores/data/test.csv
  Output: C:\Users\HP\OneDrive\Escritorio\David Guzzi\Github\DGKaggle\Playground Series\Season 6\Episode 1 - Predicting Student Test Scores/notebooks/submission_v2.csv


In [3]:
# Cargar modelo
print("Cargando modelo...")
model = CatBoostRegressor()
model.load_model(MODEL_PATH)
print(f"✓ Modelo cargado: best_catboost_model_v2.cbm")

Cargando modelo...
✓ Modelo cargado: best_catboost_model_v2.cbm


In [4]:
# Cargar datos de test
print("Cargando datos de test...")
df_test = pd.read_csv(TEST_PATH)
print(f"✓ Test: {len(df_test):,} registros")
print(f"✓ Columnas: {list(df_test.columns)}")
print(f"\nPrimeras filas:")
df_test.head()

Cargando datos de test...
✓ Test: 270,000 registros
✓ Columnas: ['id', 'age', 'gender', 'course', 'study_hours', 'class_attendance', 'internet_access', 'sleep_hours', 'sleep_quality', 'study_method', 'facility_rating', 'exam_difficulty']

Primeras filas:


,id,age,gender,course,study_hours,class_attendance,internet_access,sleep_hours,sleep_quality,study_method,facility_rating,exam_difficulty
0,630000,24,other,ba,6.85,65.2,yes,5.2,poor,group study,high,easy
1,630001,18,male,diploma,6.61,45.0,no,9.3,poor,coaching,low,easy
2,630002,24,female,b.tech,6.60,98.5,yes,6.2,good,group study,medium,moderate
3,630003,24,male,diploma,3.03,66.3,yes,5.7,average,mixed,medium,moderate
4,630004,20,female,b.tech,2.03,42.4,yes,9.2,average,coaching,low,moderate


In [5]:
# Preparar features (las mismas usadas en el entrenamiento)
selected_features = ['study_hours', 'sleep_quality', 'facility_rating',
                     'class_attendance', 'study_method']

X_test = df_test[selected_features]
print(f"✓ Features seleccionadas: {selected_features}")
print(f"✓ Shape: {X_test.shape}")
print(f"\nPrimeras filas de features:")
X_test.head()

✓ Features seleccionadas: ['study_hours', 'sleep_quality', 'facility_rating', 'class_attendance', 'study_method']
✓ Shape: (270000, 5)

Primeras filas de features:


,study_hours,sleep_quality,facility_rating,class_attendance,study_method
0,6.85,poor,high,65.2,group study
1,6.61,poor,low,45.0,coaching
2,6.60,good,medium,98.5,group study
3,3.03,average,medium,66.3,mixed
4,2.03,average,low,42.4,coaching


In [6]:
# Generar predicciones
print("Generando predicciones...")
predictions = model.predict(X_test)

print(f"\n{'='*60}")
print("ESTADÍSTICAS DE PREDICCIONES")
print(f"{'='*60}")
print(f"Registros:  {len(predictions):,}")
print(f"Media:      {np.mean(predictions):.4f}")
print(f"Std:        {np.std(predictions):.4f}")
print(f"Min:        {np.min(predictions):.4f}")
print(f"Max:        {np.max(predictions):.4f}")
print(f"{'='*60}")

Generando predicciones...

ESTADÍSTICAS DE PREDICCIONES
Registros:  270,000
Media:      62.5159
Std:        16.5364
Min:        18.7057
Max:        100.9661


In [7]:
# Crear DataFrame de submission
submission = pd.DataFrame({
    'id': df_test['id'],
    'exam_score': predictions
})

print(f"✓ Submission creado: {submission.shape}")
print(f"\nPrimeras filas:")
submission.head(10)

✓ Submission creado: (270000, 2)

Primeras filas:


,id,exam_score
0,630000,73.103743
1,630001,66.534905
2,630002,88.319549
3,630003,57.152652
4,630004,43.022871
5,630005,74.971539
6,630006,72.728202
7,630007,62.785010
8,630008,77.222688
9,630009,91.430141


In [8]:
# Verificar formato contra sample_submission
sample = pd.read_csv(SAMPLE_PATH)
print("Verificando formato...")
print(f"✓ Columnas sample:     {list(sample.columns)}")
print(f"✓ Columnas submission: {list(submission.columns)}")
print(f"✓ Registros sample:    {len(sample):,}")
print(f"✓ Registros submission:{len(submission):,}")
print(f"✓ IDs coinciden: {(submission['id'].values == sample['id'].values).all()}")

Verificando formato...
✓ Columnas sample:     ['id', 'exam_score']
✓ Columnas submission: ['id', 'exam_score']
✓ Registros sample:    270,000
✓ Registros submission:270,000
✓ IDs coinciden: True


In [9]:
# Guardar submission (NO sobrescribe sample_submission.csv)
submission.to_csv(OUTPUT_PATH, index=False)
print(f"\n{'='*60}")
print("SUBMISSION GUARDADO")
print(f"{'='*60}")
print(f"Archivo: {OUTPUT_PATH}")
print(f"Registros: {len(submission):,}")
print(f"{'='*60}")


SUBMISSION GUARDADO
Archivo: C:\Users\HP\OneDrive\Escritorio\David Guzzi\Github\DGKaggle\Playground Series\Season 6\Episode 1 - Predicting Student Test Scores/notebooks/submission_v2.csv
Registros: 270,000
